# 0.9 — Generative AI: unsupervised topic detection before CHAT ETF

**Question:** can BERTrend discover a **persistent theme** in Bloomberg all-news **before** the first generative-AI ETF (**CHAT**, inception **2023-05-17**) — with **no keyword filter, no lexicon, no embedding seed phrases** at any stage?

**Window:** 2021–2023 (baseline pre-ChatGPT + narrative burst Q4 2022 + pre-ETF 2023).

**Milestones (annotation only, not pipeline inputs):**
- ChatGPT public launch: 2022-11-30
- CHAT inception: 2023-05-17

**Outputs:** `scan_genai_allnews_{21,28}d.parquet`, `genai_stage1_parents_{21,28}d.parquet`, `genai_theme_intensity.parquet`, `genai_lead_lag.html`

Clone of [`0.7`](0.7-bertrend-granularity-scan.ipynb) all-news protocol + share intensity from [`0.8`](0.8-clean-energy-next-experiments.ipynb) + weak/strong walk from [`0.6`](0.6-clean-energy-detection.ipynb). **No Experiment B, no `clean_themes()`, no keyword baseline.** Hierarchy drill-down → [`0.10`](0.10-genai-hierarchy-drilldown.ipynb). Keyword timeline → [`0.11`](0.11-genai-keyword-timeline.ipynb). Burst-window BERTrend → [`0.12`](0.12-genai-burst-bertrend.ipynb).

In [12]:
import os, sys, lzma, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import plotly.graph_objects as go
import torch
from loguru import logger as _lg
_lg.remove(); _lg.add(sys.stderr, level="WARNING")

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("BERTREND_BASE_DIR", str(_ROOT / "notebooks" / "output" / "bertrend_base"))
RAW_DIR = _ROOT / "data" / "raw"
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from bertopic.representation import MaximalMarginalRelevance
from bertrend.BERTrend import BERTrend
from bertrend.BERTopicModel import BERTopicModel
from bertrend.utils.data_loading import (
    DOCUMENT_ID_COLUMN, SOURCE_COLUMN, TEXT_COLUMN, TIMESTAMP_COLUMN, URL_COLUMN, group_by_days,
)

# --- config ---
YEARS = [2021, 2022, 2023]
DATE_START, DATE_END = pd.Timestamp("2021-01-01"), pd.Timestamp("2023-12-31")
INCEPTION = pd.Timestamp("2023-05-17")       # CHAT ETF launch — primary gate
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")  # narrative milestone (annotation only)
ETF_TICKER = "CHAT US Equity"
ETF_XLSX = RAW_DIR / "thematic_ETFs_extract.xlsx"

BLOOMBERG_WIRES = ["BN", "BFW", "BBO"]
EMBEDDING_MODEL = "FinLang/finance-embeddings-investopedia"
DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
RANDOM_SEED = 42

GRANULARITIES = [21, 28]
PRIMARY_G = 21                             # 3-week — used for intensity / weak-strong
ALL_POOL_N = 200_000
VECTORIZER_MIN_DF = 1                      # avoid sparse-slice vectorizer failures
MIN_SIMILARITY = 0.70
MIN_ACTIVE_SLICES = 4
ALL_MIN_TOPIC, ALL_MIN_SAMPLES = 15, 5
WINDOW_SIZE = 28                           # days for classify_signals trailing window

print(f"Device {DEVICE} | window {DATE_START.date()}→{DATE_END.date()} | CHAT inception {INCEPTION.date()}")

Device mps | window 2021-01-01→2023-12-31 | CHAT inception 2023-05-17


## 1. Load Bloomberg headlines (2021–2023)

In [13]:
def strip_prefix(text: str) -> str:
    for _ in range(2):
        if ":" not in text:
            return text
        prefix, _, rest = text.partition(":")
        if not prefix or not rest or len(prefix) > 30 or len(prefix.split()) > 4:
            return text
        text = rest.strip()
    return text

frames = []
for yr in YEARS:
    with lzma.open(RAW_DIR / f"raw_news_{yr}.csv.xz", "rb") as f:
        part = (
            pl.scan_csv(f, infer_schema_length=10_000)
            .select(["Headline", "CaptureTime", "WireName"])
            .filter(pl.col("WireName").is_in(BLOOMBERG_WIRES) & pl.col("Headline").is_not_null())
            .with_columns(pl.col("CaptureTime").str.to_datetime(time_zone="UTC", strict=False))
            .collect()
        )
    frames.append(part)
    print(f"{yr}: {part.height:>9,} Bloomberg rows")

news = pl.concat(frames).to_pandas()
news["date"] = pd.to_datetime(news["CaptureTime"]).dt.tz_localize(None)
news = news[(news.date >= DATE_START) & (news.date <= DATE_END)]
news = news.dropna(subset=["Headline"]).drop_duplicates("Headline")
news["Headline"] = news["Headline"].map(strip_prefix)
news = news[news.Headline.str.split().map(len) >= 4].reset_index(drop=True)
print(f"\nTotal Bloomberg headlines: {len(news):,}")

2021: 5,889,523 Bloomberg rows
2022: 5,670,511 Bloomberg rows
2023: 5,557,627 Bloomberg rows

Total Bloomberg headlines: 2,954,113


## 2. Embedder + reusable helpers

`run_bertrend()` trains one BERTrend model per granularity (reusing precomputed embeddings); `theme_table()` summarises persistence. **No lexicon helpers.**

In [14]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

CUSTOM_STOP = list(ENGLISH_STOP_WORDS.union({
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock",
    "stocks", "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
}))

def make_df(sub: pd.DataFrame) -> pd.DataFrame:
    d = pd.DataFrame({TEXT_COLUMN: sub["Headline"].values,
                      TIMESTAMP_COLUMN: pd.to_datetime(sub["date"].values)})
    d[DOCUMENT_ID_COLUMN] = range(len(d))
    d[SOURCE_COLUMN] = "bloomberg"
    d[URL_COLUMN] = None
    return d.reset_index(drop=True)

def embed(texts_or_df) -> np.ndarray:
    texts = texts_or_df[TEXT_COLUMN].tolist() if isinstance(texts_or_df, pd.DataFrame) else texts_or_df
    return embedder.encode(texts, batch_size=64, show_progress_bar=True,
                           convert_to_numpy=True, normalize_embeddings=True)

def _slice_totals(df: pd.DataFrame, granularity: int) -> pd.Series:
    """Per-slice headline counts using the same bins as BERTrend's group_by_days."""
    return pd.Series({
        pd.Timestamp(ts).normalize(): len(g)
        for ts, g in group_by_days(df=df, day_granularity=granularity).items() if not g.empty
    }, name="slice_total")

def _bertopic(min_topic_size: int, min_samples: int,
              min_df: int = VECTORIZER_MIN_DF) -> BERTopicModel:
    cfg = f"""
[global]
language = "English"
[bertopic_model]
top_n_words = 10
verbose = false
representation_model = ["MaximalMarginalRelevance"]
zeroshot_topic_list = []
zeroshot_min_similarity = 0
[umap_model]
n_neighbors = 15
n_components = 5
min_dist = 0.0
metric = "cosine"
random_state = {RANDOM_SEED}
[hdbscan_model]
min_cluster_size = {min_topic_size}
min_samples = {min_samples}
metric = "euclidean"
cluster_selection_method = "eom"
prediction_data = true
[vectorizer_model]
ngram_range = [1, 1]
stop_words = true
min_df = {min_df}
[ctfidf_model]
bm25_weighting = false
reduce_frequent_words = true
[mmr_model]
diversity = 0.3
[reduce_outliers]
strategy = "c-tf-idf"
"""
    tm = BERTopicModel(cfg)
    tm.vectorizer_model = CountVectorizer(stop_words=CUSTOM_STOP,
                                          token_pattern=r"(?u)\b[a-zA-Z]{3,}\b",
                                          ngram_range=(1, 2), min_df=min_df)
    tm.config["bertopic_model"]["representation_model"] = [MaximalMarginalRelevance(diversity=0.4)]
    return tm

def run_bertrend(df, embeddings, granularity, min_topic_size, min_samples,
                 min_similarity=MIN_SIMILARITY):
    bt = BERTrend(topic_model=_bertopic(min_topic_size, min_samples))
    bt.config["granularity"] = granularity
    bt.config["min_similarity"] = min_similarity
    grouped = {ts: g for ts, g in group_by_days(df=df, day_granularity=granularity).items() if not g.empty}
    bt.train_topic_models(grouped_data=grouped, embedding_model=embedder, embeddings=embeddings,
                          bertrend_models_path=OUTPUT_DIR / "_genai_tmp", save_topic_models=False)
    if bt.merged_df is None:
        return None
    bt.calculate_signal_popularity()
    return bt

def theme_table(bt) -> pd.DataFrame:
    rep = {}
    for _, r in bt.merged_df.drop_duplicates("Topic").iterrows():
        x = r.get("Representation")
        rep[int(r["Topic"])] = ", ".join(x[:8]) if isinstance(x, (list, tuple)) else str(x)
    rows = []
    for tid, d in bt.topic_sizes.items():
        st = pd.to_datetime(list(d.get("Timestamps", [])))
        if len(st) == 0:
            continue
        rows.append({"theme_id": int(tid), "slices": int(st.normalize().nunique()),
                     "first_seen": st.min().normalize(), "last_seen": st.max().normalize(),
                     "docs": int(max(d.get("Docs_Count", [0]) or [0])),
                     "keywords": rep.get(int(tid), "")})
    return pd.DataFrame(rows).sort_values(["slices", "docs"], ascending=False).reset_index(drop=True)

def rep_headlines(bt, df, emb, theme_id: int, n: int = 5) -> list[str]:
    sub = bt.merged_df[bt.merged_df["Topic"] == theme_id]
    if sub.empty:
        return []
    centroid = np.asarray(sub.iloc[0]["Embedding"], dtype=float)
    centroid = centroid / (np.linalg.norm(centroid) + 1e-12)
    sims = emb @ centroid
    idx = np.argsort(-sims)[:n]
    return df.iloc[idx][TEXT_COLUMN].tolist()

def export_stage1_parents(bt, table: pd.DataFrame, granularity: int) -> Path:
    """Persist parent centroids + keywords for 0.10 hierarchy drill-down."""
    rep = {}
    emb = {}
    for _, r in bt.merged_df.drop_duplicates("Topic").iterrows():
        tid = int(r["Topic"])
        x = r.get("Representation")
        rep[tid] = ", ".join(x[:8]) if isinstance(x, (list, tuple)) else str(x)
        emb[tid] = np.asarray(r["Embedding"], dtype=float)
    rows = []
    for _, r in table.iterrows():
        tid = int(r["theme_id"])
        rows.append({
            "theme_id": tid, "granularity_days": granularity,
            "slices": int(r["slices"]), "first_seen": r["first_seen"],
            "last_seen": r["last_seen"], "docs": int(r["docs"]),
            "keywords": r["keywords"], "representation": rep.get(tid, ""),
            "embedding": emb.get(tid).tolist() if tid in emb else None,
        })
    out = pd.DataFrame(rows)
    path = OUTPUT_DIR / f"genai_stage1_parents_{granularity}d.parquet"
    out.to_parquet(path, index=False)
    return path

def theme_intensity_series(bt, df, theme_id: int, granularity: int) -> pd.DataFrame:
    slice_totals = _slice_totals(df, granularity)
    data = bt.topic_sizes.get(int(theme_id), {})
    docs = data.get("Docs_Count", [])
    ts_list = data.get("Timestamps", [])
    # one cumulative doc count per slice (last wins if duplicated)
    cum_by_slice = {}
    for i, ts in enumerate(ts_list):
        cum_by_slice[pd.Timestamp(ts).normalize()] = float(docs[i] if i < len(docs) else 0.0)
    rows = []
    prev = 0.0
    for ts in sorted(cum_by_slice):
        dc = cum_by_slice[ts]
        new_docs = max(dc - prev, 0.0)
        total = float(slice_totals.get(ts, np.nan))
        share = new_docs / total if total and total > 0 else np.nan
        rows.append({"timestamp": ts, "new_docs": new_docs, "slice_total": total,
                     "share_per10k": 10_000 * share if pd.notna(share) else np.nan})
        prev = dc
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["share_smooth"] = out["share_per10k"].rolling(3, min_periods=1).mean()
    return out

print("helpers ready")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

helpers ready


## 3. All-news BERTrend (unsupervised)

Stratified sample, embed **once**, run BERTrend at 21d and 28d. Same BERTopic params as `0.7` all-news.

In [15]:
_n_days = news["date"].dt.normalize().nunique()
_per_day = max(1, ALL_POOL_N // _n_days)
samp_all = (news.groupby(news["date"].dt.normalize(), group_keys=False)
                .apply(lambda g: g.sample(min(len(g), _per_day), random_state=RANDOM_SEED)))
df_all = make_df(samp_all.sort_values("date"))
print(f"All-news pool: {len(df_all):,} headlines (~{_per_day}/day over {_n_days} days)")

emb_all = embed(df_all)
print(f"embeddings: {emb_all.shape}")

All-news pool: 196,624 headlines (~182/day over 1094 days)


Batches:   0%|          | 0/3073 [00:00<?, ?it/s]

embeddings: (196624, 768)


In [16]:
all_results = {}

for g in GRANULARITIES:
    bt = run_bertrend(df_all, emb_all, g, ALL_MIN_TOPIC, ALL_MIN_SAMPLES)
    if bt is None:
        print(f"\nALL-NEWS · {g // 7}-WEEK: no merged themes.")
        continue
    t = theme_table(bt)
    n_slices = sum(1 for gg in group_by_days(df_all, g).values() if not gg.empty)
    pre = t[(t.slices >= MIN_ACTIVE_SLICES) & (t.first_seen < INCEPTION)]
    all_results[g] = {"bt": bt, "table": t, "pre_inception": pre, "n_slices": n_slices}
    t.assign(granularity_days=g).to_parquet(OUTPUT_DIR / f"scan_genai_allnews_{g}d.parquet")
    ppath = export_stage1_parents(bt, t, g)
    print(f"  → saved {ppath.name}")

    print(f"\n{'=' * 80}")
    print(f"ALL-NEWS · {g // 7}-WEEK · {len(t)} themes · {n_slices} slices")
    print(f"Pre-inception (slices>={MIN_ACTIVE_SLICES}, first<{INCEPTION.date()}): {len(pre)}")
    print(f"{'-' * 80}")
    for _, r in t.head(12).iterrows():
        mark = "✓" if r.first_seen < INCEPTION else " "
        flag = "★" if r.slices >= MIN_ACTIVE_SLICES else " "
        print(f"  [{mark}{flag}] T{int(r.theme_id):>3} {int(r.slices):>2} slices  "
              f"{r['first_seen'].date()}→{r['last_seen'].date()}  {r.keywords[:52]}")

2026-06-15 12:19:21,394 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-06-15 12:19:30,984 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-06-15 12:19:40,698 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF em

  → saved genai_stage1_parents_21d.parquet

ALL-NEWS · 3-WEEK · 86 themes · 53 slices
Pre-inception (slices>=4, first<2023-05-17): 83
--------------------------------------------------------------------------------
  [✓★] T  0 53 slices  2021-01-01→2023-12-29  dollar, inflation, inside, yields, markets, steady, 
  [✓★] T 34 53 slices  2021-01-01→2023-12-29  florida, sba backs, florida sba, sba, jan florida, b
  [✓★] T 27 53 slices  2021-01-01→2023-12-29  calstrs, calstrs backs, egm calstrs, proposals, back
  [✓★] T 20 53 slices  2021-01-01→2023-12-29  cut hold, cut, hold, cut sell, cut neutral, euros, n
  [✓★] T 13 53 slices  2021-01-01→2023-12-29  rated, rated buy, overweight, rated outperform, stan
  [✓★] T  4 53 slices  2021-01-01→2023-12-29  names, officer, appoints, hires, chief, sky, chairma
  [✓★] T 11 53 slices  2021-01-01→2023-12-29  premarket, shell, weekend, origin, tinto astrazeneca
  [✓★] T 12 53 slices  2021-01-01→2023-12-29  raised buy, raised, buy, raised neutral, euros

2026-06-15 12:27:38,964 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-06-15 12:27:42,829 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-06-15 12:27:46,891 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF em

  → saved genai_stage1_parents_28d.parquet

ALL-NEWS · 4-WEEK · 108 themes · 40 slices
Pre-inception (slices>=4, first<2023-05-17): 105
--------------------------------------------------------------------------------
  [✓★] T  0 40 slices  2021-01-01→2023-12-29  estimates, beats estimates, beats, outlook, forecast
  [✓★] T 40 40 slices  2021-01-01→2023-12-29  monetary, fed, feds, effects, inflation, cutting, sp
  [✓★] T  1 40 slices  2021-01-01→2023-12-29  names, chairman, officer, appoints, hires, chief, he
  [✓★] T 29 40 slices  2021-01-01→2023-12-29  florida, sba backs, sba, florida sba, jan florida, p
  [✓★] T 74 40 slices  2021-01-01→2023-12-29  inflation, december, data inside, sri lanka, sri, in
  [✓★] T 20 40 slices  2021-01-01→2023-12-29  calstrs, calstrs backs, egm calstrs, agm calstrs, pr
  [✓★] T 10 40 slices  2021-01-01→2023-12-29  rated, rated buy, rated outperform, outperform, rate
  [✓★] T  4 40 slices  2021-01-01→2023-12-29  sales, dec sales, dec, units, sales units, s

## 4. Persistence summary

Filter themes with `slices >= MIN_ACTIVE_SLICES` and `first_seen < INCEPTION`. **Qualitative review:** inspect keywords + rep headlines — no automated genAI label.

In [17]:
if PRIMARY_G not in all_results:
    raise RuntimeError(f"Primary granularity {PRIMARY_G}d failed — check §3 output.")

primary = all_results[PRIMARY_G]
t_primary = primary["table"]
pre_primary = primary["pre_inception"].copy()

print(f"Primary granularity: {PRIMARY_G}d ({PRIMARY_G // 7}-week)")
print(f"Themes before CHAT inception ({INCEPTION.date()}): {len(pre_primary)}\n")
for _, r in pre_primary.iterrows():
    print(f"  T{int(r.theme_id):>3}  {int(r.slices):>2} slices  first {r.first_seen.date()}  {r.keywords[:60]}")

# Pick candidate with max slices among pre-inception (ties: earliest first_seen)
if pre_primary.empty:
    CANDIDATE_IDS = []
    print("\n=> No theme passed persistence + pre-inception gate.")
else:
    best = pre_primary.sort_values(["slices", "first_seen"], ascending=[False, True]).iloc[0]
    CANDIDATE_IDS = [int(best.theme_id)]
    print(f"\nPrimary candidate (max persistence pre-inception): T{CANDIDATE_IDS[0]}")
    print(f"  keywords: {best.keywords}")
    for i, h in enumerate(rep_headlines(primary["bt"], df_all, emb_all, CANDIDATE_IDS[0], n=5), 1):
        print(f"  rep {i}: {h[:100]}")

Primary granularity: 21d (3-week)
Themes before CHAT inception (2023-05-17): 83

  T  0  53 slices  first 2021-01-01  dollar, inflation, inside, yields, markets, steady, gold, ra
  T 34  53 slices  first 2021-01-01  florida, sba backs, florida sba, sba, jan florida, backs, pr
  T 27  53 slices  first 2021-01-01  calstrs, calstrs backs, egm calstrs, proposals, backs, propo
  T 20  53 slices  first 2021-01-01  cut hold, cut, hold, cut sell, cut neutral, euros, neutral, 
  T 13  53 slices  first 2021-01-01  rated, rated buy, overweight, rated outperform, stanley, mor
  T  4  53 slices  first 2021-01-01  names, officer, appoints, hires, chief, sky, chairman, role
  T 11  53 slices  first 2021-01-01  premarket, shell, weekend, origin, tinto astrazeneca, icahn,
  T 12  53 slices  first 2021-01-01  raised buy, raised, buy, raised neutral, euros, buy goldman,
  T 17  52 slices  first 2021-01-01  adj, est, rev, eps, sees adj, lossshr, loss, adj eps
  T  6  52 slices  first 2021-01-01  lng, refi

## 5. Share-normalized intensity

Track `theme_share = new_docs / slice_total` for candidate theme(s). Look for ramp around ChatGPT launch (2022-11-30) and pre-ETF window.

In [18]:
bt_p = primary["bt"]
intensity_frames = []

for tid in CANDIDATE_IDS:
    ts = theme_intensity_series(bt_p, df_all, tid, PRIMARY_G)
    ts["theme_id"] = tid
    intensity_frames.append(ts)

if intensity_frames:
    intensity_all = pd.concat(intensity_frames, ignore_index=True)
    intensity_all.to_parquet(OUTPUT_DIR / "genai_theme_intensity.parquet", index=False)
    for tid in CANDIDATE_IDS:
        sub = intensity_all[intensity_all.theme_id == tid].set_index("timestamp")
        pre_inc = sub.loc[:INCEPTION, "share_smooth"].median()
        q4 = sub.loc["2022-10-01":"2023-03-31", "share_smooth"].median()
        print(f"T{tid}: pre-inception median share {pre_inc:.1f}/10k  |  Q4'22–Q1'23 median {q4:.1f}/10k")
        if pre_inc > 0:
            print(f"       ramp ratio Q4/pre-inception: {q4 / pre_inc:.1f}x")
else:
    intensity_all = pd.DataFrame()
    print("No candidate themes — skip intensity.")

T0: pre-inception median share 1054.7/10k  |  Q4'22–Q1'23 median 1227.6/10k
       ramp ratio Q4/pre-inception: 1.2x


## 6. Weak / strong signal walk-forward

At each snapshot, `classify_signals` buckets themes into noise / weak / strong. Check when **candidate** theme(s) first appear on the board before CHAT inception.

In [19]:
if not CANDIDATE_IDS:
    print("No candidates — skip weak/strong walk.")
else:
    bt_p = primary["bt"]
    cand_set = set(CANDIDATE_IDS)
    keys = sorted(pd.Timestamp(k) for k in bt_p.doc_groups.keys())
    snapshots = []
    for k in keys:
        if k > INCEPTION:
            continue
        _, w, s = bt_p.classify_signals(WINDOW_SIZE, k)
        def _ids(d):
            if d is None or d.empty or "Topic" not in d.columns:
                return []
            return sorted(cand_set & set(d["Topic"].astype(int)))
        snapshots.append((k, _ids(w), _ids(s)))

    print(f"CHAT inception: {INCEPTION.date()}  |  ChatGPT launch (ref): {CHATGPT_LAUNCH.date()}\n")
    print(f"{'snapshot':<13}{'candidate WEAK':<22}{'candidate STRONG'}")
    for k, wk, st in snapshots[-20:]:
        print(f"{str(k.date()):<13}{str(wk) if wk else '-':<22}{str(st) if st else '-'}")

    pre = [k for k, wk, st in snapshots if k <= INCEPTION and (wk or st)]
    if pre:
        first = min(pre)
        print(f"\nFirst weak/strong snapshot before inception: {first.date()}")
        print(f"  ({(INCEPTION - first).days} days before CHAT launch)")
    else:
        print("\n=> Candidate never reached weak/strong before inception at this config.")

CHAT inception: 2023-05-17  |  ChatGPT launch (ref): 2022-11-30

snapshot     candidate WEAK        candidate STRONG
2022-04-08   -                     [0]
2022-04-29   -                     [0]
2022-05-20   -                     [0]
2022-06-10   -                     [0]
2022-07-01   -                     [0]
2022-07-22   -                     [0]
2022-08-12   -                     [0]
2022-09-02   -                     [0]
2022-09-23   -                     [0]
2022-10-14   -                     [0]
2022-11-04   -                     [0]
2022-11-25   -                     [0]
2022-12-16   -                     [0]
2023-01-06   -                     [0]
2023-01-27   -                     [0]
2023-02-17   -                     [0]
2023-03-10   -                     [0]
2023-03-31   -                     [0]
2023-04-21   -                     [0]
2023-05-12   -                     [0]

First weak/strong snapshot before inception: 2021-01-01
  (866 days before CHAT launch)


## 7. CHAT ETF overlay

Parse CHAT from the local Bloomberg export; overlay theme share intensity vs price / volume / fund flows.

In [20]:
fig = go.Figure()
m = None

if not intensity_all.empty and ETF_XLSX.exists():
    ts = pd.read_excel(ETF_XLSX, sheet_name="TimeSeries", header=None)
    tickers = ts.iloc[0].ffill()
    fields = ts.iloc[1]
    cols = {f: i for i, (t, f) in enumerate(zip(tickers, fields)) if str(t) == ETF_TICKER}

    if cols:
        etf = pd.DataFrame({"date": pd.to_datetime(ts.iloc[2:, 0], errors="coerce")})
        for f, i in cols.items():
            etf[f] = pd.to_numeric(ts.iloc[2:, i].values, errors="coerce")
        etf = etf.dropna(subset=["date"]).sort_values("date")
        etf = etf[(etf.date >= DATE_START) & (etf.date <= DATE_END)]

        tid = CANDIDATE_IDS[0] if CANDIDATE_IDS else None
        if tid is not None:
            th = intensity_all[intensity_all.theme_id == tid].set_index("timestamp")
            m = th[["share_smooth"]].join(
                etf.set_index("date")[["PX_LAST", "PX_VOLUME", "FUND_FLOW"]], how="outer"
            ).sort_index()

            fig.add_scatter(x=m.index, y=m["share_smooth"], name=f"T{tid} share (smooth)",
                            line=dict(color="#d62728", width=2), yaxis="y")
            if "PX_LAST" in m.columns:
                fig.add_scatter(x=m.index, y=m["PX_LAST"], name="CHAT price",
                                line=dict(color="#1f77b4"), yaxis="y2", opacity=0.7)
            fig.add_vline(x=INCEPTION, line=dict(color="green", dash="dash"))
            fig.add_vline(x=CHATGPT_LAUNCH, line=dict(color="orange", dash="dot"))
            fig.update_layout(
                title=f"Unsupervised theme T{tid} share vs CHAT ({ETF_TICKER})",
                template="plotly_white", height=480,
                yaxis=dict(title="theme share (/10k headlines)"),
                yaxis2=dict(title="CHAT price", overlaying="y", side="right"),
            )
            fig.write_html(OUTPUT_DIR / "genai_lead_lag.html")
            print(f"Wrote {OUTPUT_DIR / 'genai_lead_lag.html'}")
            fig.show()
    else:
        print(f"{ETF_TICKER} not found in TimeSeries sheet — gate on first_seen only.")
else:
    print("Skip overlay: no intensity data or ETF xlsx missing.")

CHAT US Equity not found in TimeSeries sheet — gate on first_seen only.


## 8. Conclusions & success criteria

| Outcome | Interpretation |
|---|---|
| ≥1 theme, `slices≥4`, `first_seen < 2023-05-17`, rep docs read as genAI/LLM | Unsupervised detection **succeeded** |
| Persistent pre-inception theme but mixed AI/chips/GPU keywords | Macro blob — hierarchy drill-down needed (future) |
| No pre-inception persistent theme | GenAI too diluted in all-news (cf. clean energy `0.7`) |
| Pre-inception theme but flat share ramp | Persistence ≠ timing — share normalization insufficient |

**Validation is qualitative:** without any lexicon, identify genAI by reading keywords + rep headlines above.

In [21]:
# Summary table across granularities
rows = []
for g, res in all_results.items():
    t = res["table"]
    pre = t[(t.slices >= MIN_ACTIVE_SLICES) & (t.first_seen < INCEPTION)]
    rows.append({"granularity_d": g, "n_themes": len(t), "n_slices": res["n_slices"],
                 "pre_inception_stable": len(pre),
                 "top_pre_inc": pre.iloc[0].keywords[:50] if len(pre) else ""})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

print("\nMilestones:")
print(f"  ChatGPT launch (ref): {CHATGPT_LAUNCH.date()}")
print(f"  CHAT inception:     {INCEPTION.date()}")

 granularity_d  n_themes  n_slices  pre_inception_stable                                        top_pre_inc
            21        86        53                    83 dollar, inflation, inside, yields, markets, steady
            28       108        40                   105 estimates, beats estimates, beats, outlook, foreca

Milestones:
  ChatGPT launch (ref): 2022-11-30
  CHAT inception:     2023-05-17


## 9. Quick reload (no BERTrend re-run)

Run §0 setup, then this cell, to inspect saved parquets. Parent centroids for [`0.10`](0.10-genai-hierarchy-drilldown.ipynb): `genai_stage1_parents_{g}d.parquet`.

In [22]:
for g in GRANULARITIES:
    path = OUTPUT_DIR / f"scan_genai_allnews_{g}d.parquet"
    if not path.exists():
        print(f"missing {path.name}")
        continue
    t = pd.read_parquet(path)
    pre = t[(t.slices >= MIN_ACTIVE_SLICES) & (pd.to_datetime(t.first_seen) < INCEPTION)]
    print(f"\n{'='*72}\n{g}d · {len(t)} themes · pre-inception stable: {len(pre)}")
    for _, r in pre.head(8).iterrows():
        print(f"  T{int(r.theme_id):>3} {int(r.slices):>2} slices  {pd.Timestamp(r.first_seen).date()}  {r.keywords[:50]}")


21d · 86 themes · pre-inception stable: 83
  T  0 53 slices  2021-01-01  dollar, inflation, inside, yields, markets, steady
  T 34 53 slices  2021-01-01  florida, sba backs, florida sba, sba, jan florida,
  T 27 53 slices  2021-01-01  calstrs, calstrs backs, egm calstrs, proposals, ba
  T 20 53 slices  2021-01-01  cut hold, cut, hold, cut sell, cut neutral, euros,
  T 13 53 slices  2021-01-01  rated, rated buy, overweight, rated outperform, st
  T  4 53 slices  2021-01-01  names, officer, appoints, hires, chief, sky, chair
  T 11 53 slices  2021-01-01  premarket, shell, weekend, origin, tinto astrazene
  T 12 53 slices  2021-01-01  raised buy, raised, buy, raised neutral, euros, bu

28d · 108 themes · pre-inception stable: 105
  T  0 40 slices  2021-01-01  estimates, beats estimates, beats, outlook, foreca
  T 40 40 slices  2021-01-01  monetary, fed, feds, effects, inflation, cutting, 
  T  1 40 slices  2021-01-01  names, chairman, officer, appoints, hires, chief, 
  T 29 40 slices  2